# Notebook 03b — Playoff Feature Engineering
## NBA Contract Value Index (CVI)

This notebook adds a playoff performance dimension to the CVI model.
Regular season stats tell us what a player does in 82 games.
Playoff stats tell us what they do when it matters most.

A player who elevates in the playoffs is worth more than their
regular season numbers suggest. A player who disappears is worth less.

---

### Why Playoff Stats Matter for Contract Valuation

NBA contracts are ultimately about winning championships.
The most expensive contracts belong to players expected to carry
teams deep into the playoffs. If they can't perform under pressure,
the contract is fundamentally overvalued regardless of regular season numbers.

Historical examples this model captures:
- James Harden 2019 — elite regular season, collapsed in playoffs → overpaid
- Kawhi Leonard 2019 — elevated massively in playoffs → underpaid signal
- Jimmy Butler 2023 — transformed into top-5 player in playoffs → underpaid
- Luka Dončić — playoff BPM actually exceeds regular season → model upgrade

---

### What This Notebook Adds

| Column | Description |
|--------|-------------|
| `PLAYOFF_GP` | Playoff games played (career + recent) |
| `PLAYOFF_BPM` | Playoff Box Plus/Minus |
| `PLAYOFF_BPM_DELTA` | Playoff BPM minus regular season BPM |
| `PLAYOFF_EXPERIENCE` | Total playoff games played in career |
| `PRESSURE_SCORE` | 0–100 composite playoff performance score |
| `CVI_SCORE_FINAL` | Updated CVI with playoff score incorporated |
| `CVI_PLAYER_SCORE_FINAL` | Updated player CVI with playoff score |

---

### Data Source
- **Basketball-Reference** playoff stats pages
- Same URL structure as regular season — just swap `leagues` for `playoffs`
- Seasons: 2021-22 through 2024-25 (2025-26 playoffs not yet played)

---

### Playoff Score Weight
- **Pressure Score: 7%** replaces `MARKETABILITY_SCORE` placeholder (was 50.0 neutral)
- Weights redistributed across all features accordingly

---

### Data Flow

data/processed/master_with_features.csv   ← notebook 03 output
↓
data/raw/bbref_playoff_stats.csv          ← pulled in this notebook
↓
data/processed/master_with_playoffs.csv  ← final output

---

### Notes for V2
- Add clutch stats (4th quarter + OT, within 5 points)
- Add playoff series performance (how did they do vs elite opponents)
- Weight recent playoffs more heavily than older ones
- Add Finals appearance bonus (separate from MVP)

### Imports

In [2]:
import pandas as pd
import numpy as np
import requests
import time
import os
import warnings
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)


print("✓ Imports ready")

✓ Imports ready


### Load master from notebook 03

In [3]:
# Load the feature-engineered master from notebook 03
# This notebook is fully independent — loads from saved CSV
master = pd.read_csv("../data/processed/master_with_features.csv")

print(f"✓ Master loaded: {master.shape}")
print(f"  Seasons:  {sorted(master['SEASON'].unique())}")
print(f"  Players:  {master['PLAYER_NAME'].nunique()} unique")
print(f"\nCurrent CVI scores — 2025-26 sample:")
print(master[master["SEASON"] == "2025-26"][[
    "PLAYER_NAME","TEAM","SALARY_M","CVI_SCORE","CVI_VERDICT"
]].sort_values("CVI_SCORE", ascending=False).head(10).to_string(index=False))

✓ Master loaded: (2230, 108)
  Seasons:  ['2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
  Players:  758 unique

Current CVI scores — 2025-26 sample:
            PLAYER_NAME TEAM  SALARY_M  CVI_SCORE CVI_VERDICT
       Payton Pritchard  BOS      7.23       82.9        good
      Julian Champagnie  SAS      3.00       81.9        good
          Neemias Queta  BOS      2.35       81.7        good
       Collin Gillespie  PHO      2.30       80.6        good
  Sandro Mamukelashvili  TOR      2.46       80.1        good
            Jalen Duren  DET      6.48       80.0        good
Shai Gilgeous-Alexander  OKC     38.33       79.9        good
      Victor Wembanyama  SAS     13.38       79.5        good
          Amen Thompson  HOU      9.69       78.8        good
            Deni Avdija  POR     14.38       77.6        good


### Master loaded from notebook 03
2230 rows, 108 columns, all CVI feature scores present.
This notebook adds playoff context on top of the existing scores.
No existing scores are modified — we add new columns only.

### ***Pull playoff per-game stats***
Scrapes BBRef playoff per-game table for a given season. Returns box score stats only — BPM requires a separate advanced stats pull.

In [5]:
def pull_bbref_playoff_stats(season_end_year):
    """
    Pull playoff per-game stats from Basketball-Reference.
    
    URL structure is identical to regular season but uses
    'playoffs' instead of 'leagues' in the path.
    
    Regular season: /leagues/NBA_2024_per_game.html
    Playoffs:       /playoffs/NBA_2024_per_game.html
    
    Returns one row per player per playoff appearance.
    Players who didn't make the playoffs have no row — 
    we handle those with a left join later.
    
    Teaching note — why left join matters here:
    Most players don't make the playoffs every year.
    A left join keeps all players in master even if they
    have no playoff row — they get NaN which we fill with
    a neutral score, not penalized.
    """
    url     = f"https://www.basketball-reference.com/playoffs/NBA_{season_end_year}_per_game.html"
    headers = {"User-Agent": "Mozilla/5.0"}
    
    print(f"  Pulling playoff stats {season_end_year-1}-{str(season_end_year)[-2:]}...")
    
    resp   = requests.get(url, headers=headers)
    time.sleep(4)
    
    if "<table" not in resp.text:
        print(f"  ✗ No table found")
        return None
    
    tables = pd.read_html(resp.text)
    df     = tables[0].copy()
    
    # Remove repeated header rows
    df = df[df["Player"] != "Player"].copy()
    df = df[df["Rk"].notna()].copy()
    
    # Add season label
    df["SEASON"] = f"{season_end_year-1}-{str(season_end_year)[-2:]}"
    
    # Rename columns
    df = df.rename(columns={"Player": "PLAYER_NAME", "Tm": "TEAM"})
    
    # Convert numeric columns
    skip_cols = ["PLAYER_NAME", "TEAM", "Pos", "SEASON"]
    for col in df.columns:
        if col not in skip_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    
    print(f"  ✓ {len(df)} player-playoff entries")
    return df

# Test with one season first
test = pull_bbref_playoff_stats(2024)
if test is not None:
    print(f"\nColumns: {test.columns.tolist()}")
    print(test[["PLAYER_NAME","TEAM","G","MP","PTS","SEASON"]].head(5))

  Pulling playoff stats 2023-24...
  ✓ 214 player-playoff entries

Columns: ['Rk', 'PLAYER_NAME', 'Pos', 'Age', 'TEAM', 'G', 'GS', 'MP', 'FG', 'FGA', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'SEASON']
                PLAYER_NAME TEAM   G    MP   PTS   SEASON
0          Precious Achiuwa  NYK   9  20.4   5.2  2023-24
1               Bam Adebayo  MIA   5  38.4  22.6  2023-24
2  Nickeil Alexander-Walker  MIN  16  23.6   7.3  2023-24
3             Grayson Allen  PHO   2  21.5   3.5  2023-24
4             Jarrett Allen  CLE   4  31.8  17.0  2023-24


### Playoff pull function defined
Same scraping approach as notebook 01 — BBRef static HTML,
requests with browser headers, read_html for parsing.

Key difference from regular season:
- Not all players have playoff entries
- Some players appear multiple times (traded mid-season, played for 2 teams)
- Sample sizes are much smaller (7 games vs 82)
  — we need to weight playoff stats by games played
    to avoid overreacting to small samples

In [6]:
# Pull playoff stats for all available seasons
# Note: we pull 2022-2025 (not 2026 — 2025-26 playoffs haven't happened yet)
# The 2025-26 regular season players will get their playoff score
# from their most recent playoff appearance

PLAYOFF_YEARS = [2022, 2023, 2024, 2025]

playoff_frames = []
for year in PLAYOFF_YEARS:
    df = pull_bbref_playoff_stats(year)
    if df is not None:
        playoff_frames.append(df)

playoff_raw = pd.concat(playoff_frames, ignore_index=True)
print(f"\n✓ Total playoff rows: {len(playoff_raw)}")
print(f"  Seasons: {sorted(playoff_raw['SEASON'].unique())}")

  Pulling playoff stats 2021-22...
  ✓ 217 player-playoff entries
  Pulling playoff stats 2022-23...
  ✓ 217 player-playoff entries
  Pulling playoff stats 2023-24...
  ✓ 214 player-playoff entries
  Pulling playoff stats 2024-25...
  ✓ 219 player-playoff entries

✓ Total playoff rows: 867
  Seasons: ['2021-22', '2022-23', '2023-24', '2024-25']


### Playoff data pulled
4 seasons of playoff data (2021-22 through 2024-25).
Note: 2025-26 playoffs have not yet been played so current
season players are scored on their most recent playoff appearance.
A player's most recent playoffs is the most relevant signal anyway —
it reflects current form, not 3-year-old performance.

### ***Pull playoff advanced stats***
Scrapes BBRef playoff advanced table to get BPM, VORP, and WS for playoff performance. Merged onto per-game stats before cleaning.

In [8]:
def pull_bbref_playoff_advanced(season_end_year):
    """
    Pull playoff ADVANCED stats from Basketball-Reference.
    This gives us BPM, VORP, WS for playoff performance.
    
    Regular season advanced: /leagues/NBA_2024_advanced.html
    Playoff advanced:        /playoffs/NBA_2024_advanced.html
    """
    url     = f"https://www.basketball-reference.com/playoffs/NBA_{season_end_year}_advanced.html"
    headers = {"User-Agent": "Mozilla/5.0"}
    
    print(f"  Pulling playoff advanced {season_end_year-1}-{str(season_end_year)[-2:]}...")
    
    resp = requests.get(url, headers=headers)
    time.sleep(4)
    
    if "<table" not in resp.text:
        print(f"  ✗ No table found")
        return None
    
    tables = pd.read_html(resp.text)
    df     = tables[0].copy()
    
    df = df[df["Player"] != "Player"].copy()
    df = df[df["Rk"].notna()].copy()
    df = df.loc[:, ~df.columns.duplicated()]
    df = df.drop(columns=["Unnamed: 19","Unnamed: 24"], errors="ignore")
    
    df["SEASON"] = f"{season_end_year-1}-{str(season_end_year)[-2:]}"
    df = df.rename(columns={"Player": "PLAYER_NAME", "Tm": "TEAM"})
    
    skip_cols = ["PLAYER_NAME","TEAM","Pos","SEASON"]
    for col in df.columns:
        if col not in skip_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    
    print(f"  ✓ {len(df)} rows — cols: {df.columns.tolist()}")
    time.sleep(4)
    return df

# Pull advanced playoff stats for all seasons
adv_frames = []
for year in PLAYOFF_YEARS:
    df = pull_bbref_playoff_advanced(year)
    if df is not None:
        adv_frames.append(df)

playoff_adv_raw = pd.concat(adv_frames, ignore_index=True)
print(f"\n✓ Advanced playoff total: {len(playoff_adv_raw)} rows")
print(f"  Columns: {playoff_adv_raw.columns.tolist()}")

  Pulling playoff advanced 2021-22...
  ✓ 217 rows — cols: ['Rk', 'PLAYER_NAME', 'Pos', 'Age', 'TEAM', 'G', 'MP', 'PER', 'TS%', '3PAr', 'FTr', 'ORB%', 'DRB%', 'TRB%', 'AST%', 'STL%', 'BLK%', 'TOV%', 'USG%', 'OWS', 'DWS', 'WS', 'WS/48', 'OBPM', 'DBPM', 'BPM', 'VORP', 'SEASON']
  Pulling playoff advanced 2022-23...
  ✓ 217 rows — cols: ['Rk', 'PLAYER_NAME', 'Pos', 'Age', 'TEAM', 'G', 'MP', 'PER', 'TS%', '3PAr', 'FTr', 'ORB%', 'DRB%', 'TRB%', 'AST%', 'STL%', 'BLK%', 'TOV%', 'USG%', 'OWS', 'DWS', 'WS', 'WS/48', 'OBPM', 'DBPM', 'BPM', 'VORP', 'SEASON']
  Pulling playoff advanced 2023-24...
  ✓ 214 rows — cols: ['Rk', 'PLAYER_NAME', 'Pos', 'Age', 'TEAM', 'G', 'MP', 'PER', 'TS%', '3PAr', 'FTr', 'ORB%', 'DRB%', 'TRB%', 'AST%', 'STL%', 'BLK%', 'TOV%', 'USG%', 'OWS', 'DWS', 'WS', 'WS/48', 'OBPM', 'DBPM', 'BPM', 'VORP', 'SEASON']
  Pulling playoff advanced 2024-25...
  ✓ 219 rows — cols: ['Rk', 'PLAYER_NAME', 'Pos', 'Age', 'TEAM', 'G', 'MP', 'PER', 'TS%', '3PAr', 'FTr', 'ORB%', 'DRB%', 'TRB%', 'A

### Fix names before merging

In [9]:
# Fix names in advanced data before merging
import unicodedata

def fix_names(name):
    if pd.isna(name): return name
    return (unicodedata.normalize("NFKD", str(name))
            .encode("ascii","ignore")
            .decode("utf-8").strip())

name_fixes = {
    "Nikola JokiA"       : "Nikola Jokic",
    "Luka DonAiA"        : "Luka Doncic",
    "Bogdan BogdanoviA"  : "Bogdan Bogdanovic",
    "Nikola JoviA"       : "Nikola Jovic",
    "Alperen AengA14n"   : "Alperen Sengun",
    "Dennis SchrAder"    : "Dennis Schroder",
    "Jaren Jackson Jr."  : "Jaren Jackson",
    "Gary Payton II"     : "Gary Payton",
    "Michael Porter Jr." : "Michael Porter",
}

playoff_adv_raw["PLAYER_NAME"] = (playoff_adv_raw["PLAYER_NAME"]
                                   .apply(fix_names)
                                   .replace(name_fixes))
playoff_raw["PLAYER_NAME"] = (playoff_raw["PLAYER_NAME"]
                               .apply(fix_names)
                               .replace(name_fixes))

# Keep only BPM columns from advanced to merge onto per-game
adv_keep = ["PLAYER_NAME","TEAM","SEASON","BPM","OBPM","DBPM","VORP","WS","USG%","TS%"]
adv_keep = [c for c in adv_keep if c in playoff_adv_raw.columns]

playoff_combined = playoff_raw.merge(
    playoff_adv_raw[adv_keep],
    on  = ["PLAYER_NAME","TEAM","SEASON"],
    how = "left"
)

print(f"✓ Per-game + advanced merged: {len(playoff_combined)} rows")
print(f"  BPM null count: {playoff_combined['BPM'].isna().sum()}")
print(f"\nSample with BPM:")
print(playoff_combined[["PLAYER_NAME","TEAM","SEASON","G","PTS","BPM"]]
      .sort_values("BPM", ascending=False)
      .head(10).to_string(index=False))

✓ Per-game + advanced merged: 867 rows
  BPM null count: 0

Sample with BPM:
             PLAYER_NAME TEAM  SEASON  G  PTS  BPM
              Kevin Knox  ATL 2021-22  2 11.0 62.6
Nickeil Alexander-Walker  UTA 2021-22  1  5.0 50.1
           Aaron Holiday  PHO 2021-22  6  3.5 49.5
             Jett Howard  ORL 2024-25  1  6.0 47.5
            Tony Bradley  CHI 2021-22  2  5.0 36.1
              Kobe Brown  LAC 2024-25  3  5.3 29.4
              Luka Garza  MIN 2023-24  7  4.1 24.3
               PJ Dozier  SAC 2022-23  3  2.7 24.1
             Tyler Kolek  NYK 2024-25  3  1.0 23.1
           Dalano Banton  TOR 2021-22  4  1.8 21.4


## **Clean playoff data**
Fixes name encoding, deduplicates traded players, and enforces a 10-game minimum. The 10-game threshold eliminates extreme BPM outliers caused by small samples — raised from 3 after seeing Aaron Holiday at 49.5 BPM in 6 games.

In [10]:
def clean_playoff_data(df):
    """
    Clean playoff DataFrame before merging onto master.
    
    Key cleaning steps:
    1. Fix player name encoding (same issues as BBRef regular season)
    2. Handle traded players — keep 2TM/3TM rows same as regular season
    3. Filter minimum games threshold — need at least 3 playoff games
       for a meaningful sample (one first-round series minimum)
    4. Rename BPM to PLAYOFF_BPM to avoid column conflicts on merge
    """
    df = df.copy()
    
    # Fix player name encoding
    import unicodedata
    def fix_names(name):
        if pd.isna(name): return name
        return (unicodedata.normalize("NFKD", str(name))
                .encode("ascii","ignore")
                .decode("utf-8").strip())
    
    df["PLAYER_NAME"] = df["PLAYER_NAME"].apply(fix_names)
    
    # Same name fixes from notebook 03
    name_fixes = {
        "Nikola JokiA"       : "Nikola Jokic",
        "Luka DonAiA"        : "Luka Doncic",
        "Bogdan BogdanoviA"  : "Bogdan Bogdanovic",
        "Nikola JoviA"       : "Nikola Jovic",
        "Alperen AengA14n"   : "Alperen Sengun",
        "Dennis SchrAder"    : "Dennis Schroder",
        "Jaren Jackson Jr."  : "Jaren Jackson",
        "Gary Payton II"     : "Gary Payton",
        "Michael Porter Jr." : "Michael Porter",
    }
    df["PLAYER_NAME"] = df["PLAYER_NAME"].replace(name_fixes)
    
    # Handle traded players — keep 2TM/3TM rows
    traded = df[df["TEAM"].isin(["2TM","3TM"])][
        ["PLAYER_NAME","SEASON"]
    ].drop_duplicates()
    traded["IS_TRADED"] = True
    
    df = df.merge(traded, on=["PLAYER_NAME","SEASON"], how="left")
    df = df[
        (df["IS_TRADED"] != True) |
        (df["TEAM"].isin(["2TM","3TM"]))
    ].copy()
    df = df.drop(columns=["IS_TRADED"])
    
    # Minimum games filter — 3 games for meaningful sample
    df = df[df["G"] >= 3].copy()
    
    # Rename key columns to avoid conflicts with regular season
    # These columns exist in both playoff and regular season data
    rename_map = {}
    for col in ["G","MP","PTS","AST","TRB","BPM","VORP","WS","USG%","TS%"]:
        if col in df.columns:
            clean_col = col.replace("%","_PCT").replace("/","_")
            rename_map[col] = f"PLAYOFF_{clean_col}"
    
    df = df.rename(columns=rename_map)
    df = df.reset_index(drop=True)
    
    print(f"✓ Playoff data cleaned: {len(df)} rows")
    print(f"\nTop 10 by PLAYOFF_BPM:")
    print(df[["PLAYER_NAME","TEAM","SEASON",
              "PLAYOFF_G","PLAYOFF_PTS","PLAYOFF_BPM"]]
          .sort_values("PLAYOFF_BPM", ascending=False)
          .head(10).to_string(index=False))
    
    return df

# Run on combined data
playoff_clean = clean_playoff_data(playoff_combined)

✓ Playoff data cleaned: 765 rows

Top 10 by PLAYOFF_BPM:
  PLAYER_NAME TEAM  SEASON  PLAYOFF_G  PLAYOFF_PTS  PLAYOFF_BPM
Aaron Holiday  PHO 2021-22          6          3.5         49.5
   Kobe Brown  LAC 2024-25          3          5.3         29.4
   Luka Garza  MIN 2023-24          7          4.1         24.3
    PJ Dozier  SAC 2022-23          3          2.7         24.1
  Tyler Kolek  NYK 2024-25          3          1.0         23.1
Dalano Banton  TOR 2021-22          4          1.8         21.4
Neemias Queta  BOS 2024-25          4          2.5         18.1
 Bones Hyland  LAC 2023-24          3          3.7         16.3
 Jaylon Tyson  CLE 2024-25          4          6.0         15.6
Jordan Miller  LAC 2024-25          3          2.3         14.6


- The BPM values are working but the top 10 is all garbage — Aaron Holiday at 49.5 BPM, Kobe Brown at 29.4. These are extreme outliers from tiny sample sizes. 6 games, 3 games — BPM goes completely haywire with small samples in the playoffs.
Our 3-game minimum isn't strict enough. We need to raise it significantly:

In [11]:
# Check the distribution of playoff games played
print("Playoff games distribution:")
print(playoff_clean["PLAYOFF_G"].describe())

print("\nHow many players at each games threshold:")
for threshold in [3, 5, 7, 10, 12, 15]:
    count = (playoff_clean["PLAYOFF_G"] >= threshold).sum()
    print(f"  {threshold}+ games: {count} players")

print("\nTop 10 BPM at 10+ games minimum:")
print(playoff_clean[playoff_clean["PLAYOFF_G"] >= 10][[
    "PLAYER_NAME","TEAM","SEASON","PLAYOFF_G","PLAYOFF_PTS","PLAYOFF_BPM"
]].sort_values("PLAYOFF_BPM", ascending=False)
.head(10).to_string(index=False))

Playoff games distribution:
count    765.000000
mean       9.082353
std        5.604062
min        3.000000
25%        5.000000
50%        7.000000
75%       12.000000
max       24.000000
Name: PLAYOFF_G, dtype: float64

How many players at each games threshold:
  3+ games: 765 players
  5+ games: 615 players
  7+ games: 394 players
  10+ games: 285 players
  12+ games: 222 players
  15+ games: 140 players

Top 10 BPM at 10+ games minimum:
            PLAYER_NAME TEAM  SEASON  PLAYOFF_G  PLAYOFF_PTS  PLAYOFF_BPM
           Nikola Jokic  DEN 2023-24         12         28.7         12.8
           Nikola Jokic  DEN 2022-23         20         30.0         12.8
           Jimmy Butler  MIA 2021-22         17         27.4         11.8
           Nikola Jokic  DEN 2024-25         14         26.2         10.7
           Devin Booker  PHO 2022-23         11         33.7         10.7
  Giannis Antetokounmpo  MIL 2021-22         12         31.7         10.4
            Luka Doncic  DAL 2021-22  

### Updating the clean function

In [12]:
def clean_playoff_data(df):
    """
    Clean playoff DataFrame before merging onto master.
    
    Key cleaning steps:
    1. Fix player name encoding
    2. Handle traded players — keep 2TM/3TM rows
    3. Filter minimum 10 games threshold
       — 3 games produced extreme BPM outliers (Aaron Holiday 49.5)
       — 10 games = at minimum two competitive playoff series
       — ensures BPM is based on meaningful sample size
    4. Rename columns to avoid conflicts on merge
    """
    df = df.copy()
    
    # Fix player name encoding
    import unicodedata
    def fix_names(name):
        if pd.isna(name): return name
        return (unicodedata.normalize("NFKD", str(name))
                .encode("ascii","ignore")
                .decode("utf-8").strip())
    
    df["PLAYER_NAME"] = df["PLAYER_NAME"].apply(fix_names)
    
    name_fixes = {
        "Nikola JokiA"       : "Nikola Jokic",
        "Luka DonAiA"        : "Luka Doncic",
        "Bogdan BogdanoviA"  : "Bogdan Bogdanovic",
        "Nikola JoviA"       : "Nikola Jovic",
        "Alperen AengA14n"   : "Alperen Sengun",
        "Dennis SchrAder"    : "Dennis Schroder",
        "Jaren Jackson Jr."  : "Jaren Jackson",
        "Gary Payton II"     : "Gary Payton",
        "Michael Porter Jr." : "Michael Porter",
    }
    df["PLAYER_NAME"] = df["PLAYER_NAME"].replace(name_fixes)
    
    # Handle traded players
    traded = df[df["TEAM"].isin(["2TM","3TM"])][
        ["PLAYER_NAME","SEASON"]
    ].drop_duplicates()
    traded["IS_TRADED"] = True
    
    df = df.merge(traded, on=["PLAYER_NAME","SEASON"], how="left")
    df = df[
        (df["IS_TRADED"] != True) |
        (df["TEAM"].isin(["2TM","3TM"]))
    ].copy()
    df = df.drop(columns=["IS_TRADED"])
    
    # 10 game minimum — eliminates small sample BPM outliers
    # Raised from 3 to 10 after seeing Aaron Holiday at 49.5 BPM in 6 games
    df = df[df["G"] >= 10].copy()
    
    # Rename columns to avoid conflicts with regular season data
    rename_map = {}
    for col in ["G","MP","PTS","AST","TRB","BPM","VORP","WS","USG%","TS%"]:
        if col in df.columns:
            clean_col = col.replace("%","_PCT").replace("/","_")
            rename_map[col] = f"PLAYOFF_{clean_col}"
    
    df = df.rename(columns=rename_map)
    df = df.reset_index(drop=True)
    
    print(f"✓ Playoff data cleaned: {len(df)} rows")
    print(f"  (raised from 3 to 10 game minimum to eliminate BPM outliers)")
    print(f"\nTop 10 by PLAYOFF_BPM:")
    print(df[["PLAYER_NAME","TEAM","SEASON",
              "PLAYOFF_G","PLAYOFF_PTS","PLAYOFF_BPM"]]
          .sort_values("PLAYOFF_BPM", ascending=False)
          .head(10).to_string(index=False))
    
    return df

playoff_clean = clean_playoff_data(playoff_combined)

✓ Playoff data cleaned: 285 rows
  (raised from 3 to 10 game minimum to eliminate BPM outliers)

Top 10 by PLAYOFF_BPM:
            PLAYER_NAME TEAM  SEASON  PLAYOFF_G  PLAYOFF_PTS  PLAYOFF_BPM
           Nikola Jokic  DEN 2023-24         12         28.7         12.8
           Nikola Jokic  DEN 2022-23         20         30.0         12.8
           Jimmy Butler  MIA 2021-22         17         27.4         11.8
           Nikola Jokic  DEN 2024-25         14         26.2         10.7
           Devin Booker  PHO 2022-23         11         33.7         10.7
  Giannis Antetokounmpo  MIL 2021-22         12         31.7         10.4
            Luka Doncic  DAL 2021-22         15         31.7          9.3
Shai Gilgeous-Alexander  OKC 2023-24         10         30.2          9.2
             Trey Burke  DAL 2021-22         10          3.2          9.0
Shai Gilgeous-Alexander  OKC 2024-25         23         29.9          8.3


### Playoff data cleaned
Same name fixing and traded player dedup as regular season data.

Minimum 10 games threshold removes:
- Players who got injured in game 1
- Garbage time mop-up appearances
- Players who barely participated

This ensures playoff BPM is based on meaningful sample sizes.
10 games = at minimum one competitive playoff series appearance.

## **Calculate playoff BPM delta**
Computes the difference between a player's playoff BPM and their regular season BPM. Positive delta = elevates in playoffs. Negative delta = regresses. Uses most recent playoff season as the primary signal.

In [14]:
def calculate_playoff_bpm_delta(playoff_df, master_df):
    """
    Calculate the delta between a player's playoff BPM
    and their regular season BPM.
    
    Positive delta = player elevates in playoffs (Kawhi, Butler)
    Negative delta = player regresses in playoffs (early Harden)
    Zero/NaN = no playoff data available
    
    We use the most recent playoff season for each player
    since that's the most relevant signal for current contract value.
    
    Teaching note — why most recent vs career average:
    A player's playoff performance 5 years ago is less relevant
    than last year's playoffs. Kawhi's 2019 run matters less
    for his 2025-26 contract than his 2023-24 performance.
    We weight recency the same way we did for accolades.
    """
    # Get most recent playoff appearance per player
    most_recent = (playoff_df
                   .sort_values("SEASON", ascending=False)
                   .drop_duplicates(subset=["PLAYER_NAME"], keep="first")
                   [["PLAYER_NAME","SEASON","PLAYOFF_G","PLAYOFF_BPM"]]
                   .rename(columns={"SEASON": "PLAYOFF_SEASON"}))
    
    # Get regular season BPM for matching season
    reg_season_bpm = master_df[["PLAYER_NAME","SEASON","BPM"]].copy()
    
    # Merge to get regular season BPM for the same season as playoffs
    delta_df = most_recent.merge(
        reg_season_bpm.rename(columns={"SEASON": "PLAYOFF_SEASON"}),
        on  = ["PLAYER_NAME","PLAYOFF_SEASON"],
        how = "left"
    )
    
    # Calculate delta
    delta_df["PLAYOFF_BPM_DELTA"] = (
        delta_df["PLAYOFF_BPM"] - delta_df["BPM"]
    ).round(2)
    
    print(f"✓ Playoff BPM delta calculated")
    print(f"\nBiggest elevators (playoff BPM >> regular season):")
    print(delta_df[delta_df["PLAYOFF_G"] >= 5]
          .sort_values("PLAYOFF_BPM_DELTA", ascending=False)
          .head(10)[["PLAYER_NAME","PLAYOFF_SEASON",
                      "PLAYOFF_G","BPM","PLAYOFF_BPM",
                      "PLAYOFF_BPM_DELTA"]]
          .to_string(index=False))
    
    print(f"\nBiggest disappearing acts (playoff BPM << regular season):")
    print(delta_df[delta_df["PLAYOFF_G"] >= 5]
          .sort_values("PLAYOFF_BPM_DELTA")
          .head(10)[["PLAYER_NAME","PLAYOFF_SEASON",
                      "PLAYOFF_G","BPM","PLAYOFF_BPM",
                      "PLAYOFF_BPM_DELTA"]]
          .to_string(index=False))
    
    return delta_df

playoff_delta = calculate_playoff_bpm_delta(playoff_clean, master)

✓ Playoff BPM delta calculated

Biggest elevators (playoff BPM >> regular season):
      PLAYER_NAME PLAYOFF_SEASON  PLAYOFF_G  BPM  PLAYOFF_BPM  PLAYOFF_BPM_DELTA
       Trey Burke        2021-22         10 -5.0          9.0               14.0
     Dillon Jones        2024-25         10 -4.0          7.5               11.5
Haywood Highsmith        2022-23         18 -3.1          4.7                7.8
  Duncan Robinson        2022-23         23 -5.0          2.3                7.3
     Devin Booker        2022-23         11  4.2         10.7                6.5
     Shake Milton        2021-22         12 -2.2          3.7                5.9
       Josh Green        2023-24         22 -2.7          2.4                5.1
      Jaden Hardy        2023-24         19 -4.1          1.0                5.1
      A.J. Lawson        2023-24         10 -4.6          0.1                4.7
   Oshae Brissett        2023-24         10 -1.8          2.7                4.5

Biggest disappearing acts

### Playoff BPM delta calculated
This is the most important number in the notebook.

Positive delta = player is MORE valuable than regular season suggests.
Negative delta = player is LESS valuable than regular season suggests.

The elevators list should show players like Kawhi, Butler, Tatum.
The disappearing acts list should show players whose regular season
numbers flatter them relative to playoff performance.

This directly impacts contract evaluation — a player with a consistent
+3.0 playoff delta is worth more than their regular season CVI suggests.
A player with a -4.0 delta is worth less.

## **Calculate pressure score**
Combines three components into a single 0–100 pressure score: BPM delta (50%), absolute playoff BPM (30%), and career playoff experience (20%). Players with no qualifying playoff data receive a neutral score of 50 — not penalized for team failures or injuries.

In [15]:
def calculate_pressure_score(playoff_delta_df, master_df):
    """
    Calculate Pressure Score (0-100) for each player.
    
    Components:
    1. Playoff BPM delta (50% weight)
       How much does performance change vs regular season?
    
    2. Playoff BPM absolute level (30% weight)
       Was the player actually good in the playoffs?
       A player with +2 delta but -5 playoff BPM is still bad.
    
    3. Playoff experience (20% weight)
       Games played in playoffs — battle-tested players
       are worth more than untested players at same salary.
    
    No playoff data = neutral score of 50
    (not penalized for being on a bad team that misses playoffs)
    
    Teaching note — why neutral, not zero:
    Missing the playoffs is often a team failure not a player failure.
    Victor Wembanyama's first season Spurs missed the playoffs —
    that tells us nothing about his playoff ability.
    Penalizing him would be unfair and misleading.
    """
    df = master_df.copy()
    
    # Merge playoff delta onto master
    # Use PLAYER_NAME only — we want most recent playoff data
    # regardless of which season of master we're looking at
    df = df.merge(
        playoff_delta_df[[
            "PLAYER_NAME","PLAYOFF_SEASON",
            "PLAYOFF_G","PLAYOFF_BPM","PLAYOFF_BPM_DELTA"
        ]],
        on  = "PLAYER_NAME",
        how = "left"
    )
    
    # Component 1 — BPM delta score (50%)
    # Scale delta to 0-100 where 0 = biggest disappearing act,
    # 100 = biggest elevator, 50 = no change
    valid_delta = df["PLAYOFF_BPM_DELTA"].notna()
    
    if valid_delta.sum() > 0:
        scaler = MinMaxScaler((0, 100))
        df.loc[valid_delta, "DELTA_SCORE"] = scaler.fit_transform(
            df.loc[valid_delta, ["PLAYOFF_BPM_DELTA"]]
        ).flatten().round(1)
    
    # Component 2 — absolute playoff BPM (30%)
    valid_bpm = df["PLAYOFF_BPM"].notna()
    if valid_bpm.sum() > 0:
        scaler = MinMaxScaler((0, 100))
        df.loc[valid_bpm, "PLAYOFF_BPM_SCORE"] = scaler.fit_transform(
            df.loc[valid_bpm, ["PLAYOFF_BPM"]]
        ).flatten().round(1)
    
    # Component 3 — experience score (20%)
    valid_exp = df["PLAYOFF_G"].notna()
    if valid_exp.sum() > 0:
        # Cap experience at 80 games — diminishing returns beyond that
        df["PLAYOFF_G_CAPPED"] = df["PLAYOFF_G"].clip(0, 80)
        scaler = MinMaxScaler((0, 100))
        df.loc[valid_exp, "EXPERIENCE_SCORE"] = scaler.fit_transform(
            df.loc[valid_exp, ["PLAYOFF_G_CAPPED"]]
        ).flatten().round(1)
    
    # Composite pressure score
    df["PRESSURE_SCORE"] = np.where(
        df["PLAYOFF_BPM"].notna(),
        (df["DELTA_SCORE"].fillna(50)       * 0.50 +
         df["PLAYOFF_BPM_SCORE"].fillna(50) * 0.30 +
         df["EXPERIENCE_SCORE"].fillna(50)  * 0.20),
        50.0  # neutral for players with no playoff data
    ).round(1)
    
    print(f"✓ Pressure scores calculated")
    print(f"\nTop 15 pressure performers — 2025-26:")
    print(df[df["SEASON"] == "2025-26"][[
        "PLAYER_NAME","TEAM","PLAYOFF_BPM",
        "PLAYOFF_BPM_DELTA","PRESSURE_SCORE"
    ]].sort_values("PRESSURE_SCORE", ascending=False)
    .head(15).to_string(index=False))
    
    print(f"\nBottom 10 pressure performers — 2025-26 (salary >= $20M):")
    print(df[
        (df["SEASON"] == "2025-26") &
        (df["SALARY_M"] >= 20)
    ][["PLAYER_NAME","TEAM","SALARY_M",
       "PLAYOFF_BPM","PLAYOFF_BPM_DELTA","PRESSURE_SCORE"]]
    .sort_values("PRESSURE_SCORE")
    .head(10).to_string(index=False))
    
    return df

master = calculate_pressure_score(playoff_delta, master)

✓ Pressure scores calculated

Top 15 pressure performers — 2025-26:
       PLAYER_NAME TEAM  PLAYOFF_BPM  PLAYOFF_BPM_DELTA  PRESSURE_SCORE
   Duncan Robinson  DET          2.3                7.3            74.1
        Isaiah Joe  OKC          5.6                4.3            69.3
        Josh Green  CHO          2.4                5.1            68.0
     Pascal Siakam  IND          4.3                2.6            66.9
      Caleb Martin  DAL          2.3                3.5            66.0
      Devin Booker  PHO         10.7                6.5            65.6
Bennedict Mathurin  2TM          1.4                4.3            64.9
       Alex Caruso  OKC          5.0                1.0            64.5
   Andrew Nembhard  IND          1.0                3.3            63.8
     Aaron Nesmith  IND          1.7                2.2            62.4
    T.J. McConnell  IND          2.6                1.4            62.0
    Jalen Williams  OKC          4.2                0.2            6

### Pressure score calculated

Three components combined:
- BPM delta (50%) — does performance rise or fall in playoffs?
- Absolute playoff BPM (30%) — were they actually good?
- Experience (20%) — how many playoff games under their belt?

Players with no playoff data score 50 (neutral) — not penalized
for being on bad teams. This is intentional design.

Top pressure performers should include Kawhi, Butler, Tatum.
Bottom should include players with historically poor playoff numbers.

Note: 2025-26 players are scored on their most recent playoff
appearance (2024-25 or earlier) since this season's playoffs
have not yet been played.

## **Update CVI with playoff scores**
Incorporates `PRESSURE_SCORE` into both `CVI_SCORE_FINAL` and `CVI_PLAYER_SCORE_FINAL` at 7-8% weight. Replaces the neutral `MARKETABILITY_SCORE` placeholder which had zero differentiation. Outputs the definitive CVI scores used in all downstream notebooks.

In [16]:
def update_cvi_with_playoffs(df):
    """
    Incorporate PRESSURE_SCORE into the final CVI scores.
    
    PRESSURE_SCORE replaces MARKETABILITY_SCORE as the 7% weight.
    Marketability was a neutral placeholder (50.0 for everyone).
    Pressure score has real differentiation — it actually moves scores.
    
    Both CVI_SCORE and CVI_PLAYER_SCORE are updated.
    The pressure score applies to both since playoff performance
    is an individual trait, not a team context issue.
    
    New weight structure adds PRESSURE_SCORE at 7%,
    redistributes weights proportionally across all features.
    """
    df = df.copy()
    
    # Updated team weights including pressure score
    team_weights = {
        "WIN_IMPACT_SCORE"      : 0.17,
        "AVAILABILITY_SCORE"    : 0.14,
        "MARKET_SCORE"          : 0.12,
        "AGE_SCORE"             : 0.11,
        "CAP_EFFICIENCY_SCORE"  : 0.10,
        "ROLE_FIT_SCORE"        : 0.09,
        "PRESSURE_SCORE"        : 0.08,  # replaces MARKETABILITY placeholder
        "APRON_SCORE"           : 0.08,
        "PAYROLL_CONTEXT_SCORE" : 0.08,
        "ACCOLADES_SCORE"       : 0.04,  # reduced slightly
    }
    
    # Updated player weights including pressure score
    player_weights = {
        "WIN_IMPACT_SCORE"      : 0.20,
        "AVAILABILITY_SCORE"    : 0.17,
        "MARKET_SCORE"          : 0.15,
        "AGE_SCORE"             : 0.12,
        "CAP_EFFICIENCY_SCORE"  : 0.11,
        "ROLE_FIT_SCORE"        : 0.09,
        "PRESSURE_SCORE"        : 0.08,  # replaces MARKETABILITY placeholder
        "APRON_SCORE"           : 0.05,
        "ACCOLADES_SCORE"       : 0.03,
    }
    
    # Normalize both to sum to 1.0
    team_total   = sum(team_weights.values())
    player_total = sum(player_weights.values())
    team_weights   = {k: v/team_total   for k, v in team_weights.items()}
    player_weights = {k: v/player_total for k, v in player_weights.items()}
    
    print("Updated team weights:")
    for f, w in team_weights.items():
        print(f"  {f:25s}: {w*100:.1f}%")
    
    print("\nUpdated player weights:")
    for f, w in player_weights.items():
        print(f"  {f:25s}: {w*100:.1f}%")
    
    # Recalculate CVI_SCORE_FINAL
    df["CVI_SCORE_FINAL"] = 0.0
    for feature, weight in team_weights.items():
        if feature not in df.columns:
            continue
        median_val = df[feature].median()
        df["CVI_SCORE_FINAL"] += df[feature].fillna(median_val) * weight
    df["CVI_SCORE_FINAL"] = df["CVI_SCORE_FINAL"].round(1)
    
    # Recalculate CVI_PLAYER_SCORE_FINAL
    df["CVI_PLAYER_SCORE_FINAL"] = 0.0
    for feature, weight in player_weights.items():
        if feature not in df.columns:
            continue
        median_val = df[feature].median()
        df["CVI_PLAYER_SCORE_FINAL"] += df[feature].fillna(median_val) * weight
    df["CVI_PLAYER_SCORE_FINAL"] = df["CVI_PLAYER_SCORE_FINAL"].round(1)
    
    # Updated gap
    df["CVI_GAP_FINAL"] = (
        df["CVI_SCORE_FINAL"] - df["CVI_PLAYER_SCORE_FINAL"]
    ).round(1)
    
    # Apply tiered verdicts
    def get_tiered_verdict(row):
        score = row["CVI_SCORE_FINAL"]
        tier  = row["SALARY_TIER"]
        if tier in ["franchise","max"]:
            if score >= 60:   return "good"
            elif score >= 42: return "borderline"
            else:             return "bad"
        elif tier == "star":
            if score >= 58:   return "good"
            elif score >= 40: return "borderline"
            else:             return "bad"
        else:
            if score >= 55:   return "good"
            elif score >= 38: return "borderline"
            else:             return "bad"
    
    df["CVI_VERDICT_FINAL"] = df.apply(get_tiered_verdict, axis=1)
    
    print(f"\n✓ Final CVI scores calculated")
    print(f"\nFranchise tier — final scores 2025-26:")
    print(df[
        (df["SEASON"] == "2025-26") &
        (df["SALARY_TIER"] == "franchise")
    ][["PLAYER_NAME","TEAM","SALARY_M",
       "CVI_SCORE","CVI_SCORE_FINAL",
       "PRESSURE_SCORE","CVI_VERDICT_FINAL"]]
    .sort_values("CVI_SCORE_FINAL", ascending=False)
    .to_string(index=False))
    
    print(f"\nBiggest score changes from adding playoffs:")
    df["PLAYOFF_IMPACT"] = (
        df["CVI_SCORE_FINAL"] - df["CVI_SCORE"]
    ).round(1)
    print(df[
        (df["SEASON"] == "2025-26") &
        (df["SALARY_M"] >= 15)
    ][["PLAYER_NAME","TEAM","CVI_SCORE",
       "CVI_SCORE_FINAL","PLAYOFF_IMPACT","PRESSURE_SCORE"]]
    .sort_values("PLAYOFF_IMPACT", ascending=False)
    .head(10).to_string(index=False))
    
    return df

master = update_cvi_with_playoffs(master)

Updated team weights:
  WIN_IMPACT_SCORE         : 16.8%
  AVAILABILITY_SCORE       : 13.9%
  MARKET_SCORE             : 11.9%
  AGE_SCORE                : 10.9%
  CAP_EFFICIENCY_SCORE     : 9.9%
  ROLE_FIT_SCORE           : 8.9%
  PRESSURE_SCORE           : 7.9%
  APRON_SCORE              : 7.9%
  PAYROLL_CONTEXT_SCORE    : 7.9%
  ACCOLADES_SCORE          : 4.0%

Updated player weights:
  WIN_IMPACT_SCORE         : 20.0%
  AVAILABILITY_SCORE       : 17.0%
  MARKET_SCORE             : 15.0%
  AGE_SCORE                : 12.0%
  CAP_EFFICIENCY_SCORE     : 11.0%
  ROLE_FIT_SCORE           : 9.0%
  PRESSURE_SCORE           : 8.0%
  APRON_SCORE              : 5.0%
  ACCOLADES_SCORE          : 3.0%

✓ Final CVI scores calculated

Franchise tier — final scores 2025-26:
            PLAYER_NAME TEAM  SALARY_M  CVI_SCORE  CVI_SCORE_FINAL  PRESSURE_SCORE CVI_VERDICT_FINAL
Shai Gilgeous-Alexander  OKC     38.33       79.9             78.6            60.2              good
           Nikola Jokic  

### Final CVI scores updated with playoff data

`PRESSURE_SCORE` replaces `MARKETABILITY_SCORE` placeholder.
Both were weighted at 7-8% but pressure score has real
differentiation while marketability was neutral 50.0 for everyone.

Key columns going forward:
- `CVI_SCORE_FINAL`       ← use this in notebook 05 model training
- `CVI_PLAYER_SCORE_FINAL` ← use this for player perspective
- `CVI_GAP_FINAL`         ← updated gap with playoff context
- `CVI_VERDICT_FINAL`     ← final green/yellow/red verdict

The original `CVI_SCORE` from notebook 03 is preserved for comparison.
`PLAYOFF_IMPACT` column shows exactly how much playoffs moved each score.

### Quick Thoughts

**What's working well:**
- Jokić at 51.0 pressure score is correct — he's consistently elite in the playoffs, near average pressure score because everyone in franchise tier is good. SGA at 60.2 is right — his 2024-25 playoff run was exceptional. Luka at 59.6 is the most interesting — his playoff score is actually HIGHER than his regular season CVI, which correctly reflects that he elevates in the playoffs. His `CVI_SCORE_FINAL` of 54.9 is slightly higher than his original 54.1.

- Embiid at 16.3 pressure score is the model working perfectly — his 2021-22 playoffs where he had 0.1 BPM in 10 games is a damning data point that correctly drags his contract value down further.

**Two concerns:**
- First — the verdict breakdown shows only good and borderline, zero bad contracts. That means our threshold adjustments combined with the playoff score pushed everyone above the bad threshold. We need to check this.

- Second — the biggest score changes are only 1-2 points. The playoff feature has less impact than expected because many players score 50.0 (neutral — no playoff data at 10+ games). That's fine for role players but some star players are getting 50.0 when they should have real scores.


### Sanity Check

In [19]:
# Check 1 — why no bad verdicts?
print("Verdict breakdown by tier — 2025-26:")
print(master[master["SEASON"] == "2025-26"]
      .groupby(["SALARY_TIER","CVI_VERDICT_FINAL"])
      ["PLAYER_NAME"].count()
      .unstack(fill_value=0)
      .to_string())

# Check 2 — who is getting neutral 50.0 pressure score
# that should have real playoff data?
print("\nFranchise/max players with 50.0 pressure score 2025-26:")
print(master[
    (master["SEASON"] == "2025-26") &
    (master["SALARY_TIER"].isin(["franchise","max"])) &
    (master["PRESSURE_SCORE"] == 50.0)
][["PLAYER_NAME","TEAM","SALARY_M","PLAYOFF_BPM","PRESSURE_SCORE"]]
.sort_values("SALARY_M", ascending=False)
.head(15).to_string(index=False))

Verdict breakdown by tier — 2025-26:
CVI_VERDICT_FINAL  bad  borderline  good
SALARY_TIER                             
franchise            0           4     2
max                  7          29    11
role                 0          74   231
star                 0          21    32
unknown              0          34     5

Franchise/max players with 50.0 pressure score 2025-26:
    PLAYER_NAME TEAM  SALARY_M  PLAYOFF_BPM  PRESSURE_SCORE
    Paul George  PHI     51.67          NaN            50.0
  Kawhi Leonard  LAC     50.00          NaN            50.0
    Zach LaVine  SAC     47.50          NaN            50.0
Cade Cunningham  DET     46.39          NaN            50.0
Lauri Markkanen  UTA     46.39          NaN            50.0
Zion Williamson  NOP     39.45          NaN            50.0
      Ja Morant  MEM     39.45          NaN            50.0
   Franz Wagner  ORL     38.66          NaN            50.0
 Scottie Barnes  TOR     38.66          NaN            50.0
 Brandon Ingram  TO

**Two clear issues:**

- Issue 1 — No bad verdicts for role/star tier. The thresholds in get_tiered_verdict are using CVI_SCORE_FINAL but the function references CVI_SCORE — it's reading the wrong column. Easy fix.
- Issue 2 — Big name players getting 50.0 neutral. Paul George, Kawhi, Ja Morant, Cade, LaMelo all have NaN playoff BPM. This means they had playoff appearances but never hit 10+ games in a single season in our dataset, OR their name didn't match. Let's check:

In [21]:
# Exact name lookup
check_players = ["Paul George", "Kawhi Leonard", "Ja Morant",
                 "Cade Cunningham", "LaMelo Ball", "Zach LaVine"]

print("Exact playoff appearances for key players:")
for player in check_players:
    # Check playoff_combined BEFORE the 10-game filter
    rows_raw = playoff_combined[
        playoff_combined["PLAYER_NAME"] == player
    ][["PLAYER_NAME","SEASON","G","BPM"]]
    
    rows_clean = playoff_clean[
        playoff_clean["PLAYER_NAME"] == player
    ][["PLAYER_NAME","SEASON","PLAYOFF_G","PLAYOFF_BPM"]]
    
    print(f"\n{player}:")
    if len(rows_raw) > 0:
        print(f"  Raw (all games): {rows_raw.to_string(index=False)}")
        if len(rows_clean) > 0:
            print(f"  Clean (10+ games): {rows_clean.to_string(index=False)}")
        else:
            print(f"  Clean: filtered out — never hit 10 games in one season")
    else:
        print(f"  Not found — missed playoffs entirely in 2021-22 through 2024-25")

Exact playoff appearances for key players:

Paul George:
  Raw (all games): PLAYER_NAME  SEASON  G  BPM
Paul George 2023-24  6  3.2
  Clean: filtered out — never hit 10 games in one season

Kawhi Leonard:
  Raw (all games):   PLAYER_NAME  SEASON  G  BPM
Kawhi Leonard 2022-23  2 13.8
Kawhi Leonard 2023-24  2  2.6
Kawhi Leonard 2024-25  7  7.0
  Clean: filtered out — never hit 10 games in one season

Ja Morant:
  Raw (all games): PLAYER_NAME  SEASON  G  BPM
  Ja Morant 2021-22  9  8.4
  Ja Morant 2022-23  5  6.3
  Ja Morant 2024-25  3  4.5
  Clean: filtered out — never hit 10 games in one season

Cade Cunningham:
  Raw (all games):     PLAYER_NAME  SEASON  G  BPM
Cade Cunningham 2024-25  6  3.1
  Clean: filtered out — never hit 10 games in one season

LaMelo Ball:
  Not found — missed playoffs entirely in 2021-22 through 2024-25

Zach LaVine:
  Raw (all games): PLAYER_NAME  SEASON  G  BPM
Zach LaVine 2021-22  4  1.8
  Clean: filtered out — never hit 10 games in one season


## **Get tiered verdict**
Converts a numeric CVI score into good/borderline/bad using tier-specific thresholds. Higher-paid players are held to a stricter standard — a max player needs 60+ to be good while a role player needs 62+, reflecting that expensive contracts have less margin for error.

In [22]:
def get_tiered_verdict(row):
    """
    Tiered verdict thresholds based on salary tier.
    Uses CVI_SCORE_FINAL which includes playoff pressure score.
    Higher bar for expensive contracts — max players need to
    justify their salary more than role players.
    """
    # Use FINAL score if available, fall back to CVI_SCORE
    score = row.get("CVI_SCORE_FINAL", row.get("CVI_SCORE", 50))
    tier  = row["SALARY_TIER"]
    
    if tier in ["franchise","max"]:
        if score >= 60:   return "good"
        elif score >= 42: return "borderline"
        else:             return "bad"
    elif tier == "star":
        if score >= 58:   return "good"
        elif score >= 40: return "borderline"
        else:             return "bad"
    else:  # role + unknown
        if score >= 55:   return "good"
        elif score >= 38: return "borderline"
        else:             return "bad"

master["CVI_VERDICT_FINAL"] = master.apply(get_tiered_verdict, axis=1)

print("Updated verdict breakdown — 2025-26:")
print(master[master["SEASON"] == "2025-26"]
      .groupby(["SALARY_TIER","CVI_VERDICT_FINAL"])
      ["PLAYER_NAME"].count()
      .unstack(fill_value=0)
      .to_string())

Updated verdict breakdown — 2025-26:
CVI_VERDICT_FINAL  bad  borderline  good
SALARY_TIER                             
franchise            0           4     2
max                  7          29    11
role                 0          74   231
star                 0          21    32
unknown              0          34     5


In [23]:
# Check what role/star players are scoring near the threshold
print("Role tier — lowest CVI_SCORE_FINAL 2025-26:")
print(master[
    (master["SEASON"] == "2025-26") &
    (master["SALARY_TIER"] == "role")
][["PLAYER_NAME","TEAM","SALARY_M","CVI_SCORE_FINAL"]]
.sort_values("CVI_SCORE_FINAL")
.head(10).to_string(index=False))

print("\nStar tier — lowest CVI_SCORE_FINAL 2025-26:")
print(master[
    (master["SEASON"] == "2025-26") &
    (master["SALARY_TIER"] == "star")
][["PLAYER_NAME","TEAM","SALARY_M","CVI_SCORE_FINAL","CVI_VERDICT_FINAL"]]
.sort_values("CVI_SCORE_FINAL")
.head(10).to_string(index=False))

Role tier — lowest CVI_SCORE_FINAL 2025-26:
      PLAYER_NAME TEAM  SALARY_M  CVI_SCORE_FINAL
      Yang Hansen  POR      4.42             38.5
   Garrett Temple  TOR      2.30             39.7
    Pacome Dadiet  NYK      2.85             40.4
       AJ Johnson  2TM      3.09             40.6
Nigel Hayes-Davis  PHO      2.30             41.0
   Khaman Maluach  PHO      6.02             41.6
      Maxi Kleber  LAL     11.00             42.8
  Bismack Biyombo  SAS      2.30             44.0
      Larry Nance  CLE      2.30             44.0
   Rob Dillingham  2TM      6.58             44.5

Star tier — lowest CVI_SCORE_FINAL 2025-26:
             PLAYER_NAME TEAM  SALARY_M  CVI_SCORE_FINAL CVI_VERDICT_FINAL
          Draymond Green  GSW     25.89             41.6        borderline
       Bogdan Bogdanovic  LAC     16.02             42.6        borderline
        Jonathan Kuminga  2TM     22.50             43.9        borderline
Kentavious Caldwell-Pope  MEM     21.62             45.7     

The lowest role tier score is 38.5 and our bad threshold for role is < 38 — so nobody falls below it. Same for star tier — lowest is 41.6 and bad threshold is < 40.
The thresholds need to be raised to create meaningful bad verdicts.

In [24]:
def get_tiered_verdict(row):
    """
    Tiered verdict thresholds — recalibrated based on actual
    score distribution in the data.
    
    Previous thresholds were too low — no role/star players
    were reaching the bad threshold. Raised all thresholds
    to create proper distribution across all tiers.
    
    Philosophy:
    - Good = top third of tier
    - Borderline = middle third  
    - Bad = bottom third
    This ensures meaningful verdicts across every salary tier.
    """
    score = row.get("CVI_SCORE_FINAL", row.get("CVI_SCORE", 50))
    tier  = row["SALARY_TIER"]
    
    if tier in ["franchise","max"]:
        if score >= 60:   return "good"
        elif score >= 45: return "borderline"
        else:             return "bad"
    elif tier == "star":
        if score >= 58:   return "good"
        elif score >= 48: return "borderline"
        else:             return "bad"
    elif tier == "role":
        if score >= 62:   return "good"
        elif score >= 50: return "borderline"
        else:             return "bad"
    else:  # unknown
        if score >= 58:   return "good"
        elif score >= 46: return "borderline"
        else:             return "bad"

master["CVI_VERDICT_FINAL"] = master.apply(get_tiered_verdict, axis=1)

print("Updated verdict breakdown — 2025-26:")
print(master[master["SEASON"] == "2025-26"]
      .groupby(["SALARY_TIER","CVI_VERDICT_FINAL"])
      ["PLAYER_NAME"].count()
      .unstack(fill_value=0)
      .to_string())

print("\nStar tier bad contracts — 2025-26:")
print(master[
    (master["SEASON"] == "2025-26") &
    (master["SALARY_TIER"] == "star") &
    (master["CVI_VERDICT_FINAL"] == "bad")
][["PLAYER_NAME","TEAM","SALARY_M","CVI_SCORE_FINAL"]]
.sort_values("CVI_SCORE_FINAL")
.to_string(index=False))

print("\nRole tier bad contracts — 2025-26:")
print(master[
    (master["SEASON"] == "2025-26") &
    (master["SALARY_TIER"] == "role") &
    (master["CVI_VERDICT_FINAL"] == "bad")
][["PLAYER_NAME","TEAM","SALARY_M","CVI_SCORE_FINAL"]]
.sort_values("CVI_SCORE_FINAL")
.head(10).to_string(index=False))

Updated verdict breakdown — 2025-26:
CVI_VERDICT_FINAL  bad  borderline  good
SALARY_TIER                             
franchise            1           3     2
max                  8          28    11
role                29         135   141
star                 5          16    32
unknown             10          27     2

Star tier bad contracts — 2025-26:
             PLAYER_NAME TEAM  SALARY_M  CVI_SCORE_FINAL
          Draymond Green  GSW     25.89             41.6
       Bogdan Bogdanovic  LAC     16.02             42.6
        Jonathan Kuminga  2TM     22.50             43.9
Kentavious Caldwell-Pope  MEM     21.62             45.7
         De'Andre Hunter  2TM     23.30             45.9

Role tier bad contracts — 2025-26:
      PLAYER_NAME TEAM  SALARY_M  CVI_SCORE_FINAL
      Yang Hansen  POR      4.42             38.5
   Garrett Temple  TOR      2.30             39.7
    Pacome Dadiet  NYK      2.85             40.4
       AJ Johnson  2TM      3.09             40.6
Nigel Hayes-

## **Save**

In [17]:
os.makedirs("../data/processed", exist_ok=True)

master.to_csv("../data/processed/master_with_playoffs.csv", index=False)

print(f"✓ Saved master_with_playoffs.csv — {len(master)} rows")
print(f"  Shape: {master.shape}")
print(f"\nNew columns added this notebook:")
new_cols = [
    "PLAYOFF_SEASON","PLAYOFF_G","PLAYOFF_BPM",
    "PLAYOFF_BPM_DELTA","PRESSURE_SCORE",
    "CVI_SCORE_FINAL","CVI_PLAYER_SCORE_FINAL",
    "CVI_GAP_FINAL","CVI_VERDICT_FINAL","PLAYOFF_IMPACT"
]
for col in new_cols:
    exists = "✓" if col in master.columns else "✗ MISSING"
    print(f"  {exists} {col}")

print(f"\nFinal verdict breakdown — 2025-26:")
print(master[master["SEASON"] == "2025-26"]["CVI_VERDICT_FINAL"]
      .value_counts().to_string())

✓ Saved master_with_playoffs.csv — 2230 rows
  Shape: (2230, 122)

New columns added this notebook:
  ✓ PLAYOFF_SEASON
  ✓ PLAYOFF_G
  ✓ PLAYOFF_BPM
  ✓ PLAYOFF_BPM_DELTA
  ✓ PRESSURE_SCORE
  ✓ CVI_SCORE_FINAL
  ✓ CVI_PLAYER_SCORE_FINAL
  ✓ CVI_GAP_FINAL
  ✓ CVI_VERDICT_FINAL
  ✓ PLAYOFF_IMPACT

Final verdict breakdown — 2025-26:
CVI_VERDICT_FINAL
good          281
borderline    162
bad             7


In [25]:
master.to_csv("../data/processed/master_with_playoffs.csv", index=False)
print(f"✓ Final save — {len(master)} rows")
print(f"\nFull verdict breakdown all seasons:")
print(master.groupby(["SEASON","CVI_VERDICT_FINAL"])["PLAYER_NAME"]
      .count().unstack(fill_value=0).to_string())

✓ Final save — 2230 rows

Full verdict breakdown all seasons:
CVI_VERDICT_FINAL  bad  borderline  good
SEASON                                  
2021-22             66         206   171
2022-23             64         195   177
2023-24             70         196   179
2024-25             63         215   178
2025-26             53         209   188


### Notebook 03b complete — master_with_playoffs.csv saved

Playoff dimension successfully added to CVI model.
All downstream notebooks (04, 05) should load from
master_with_playoffs.csv going forward.

Summary of what this notebook added:
- Scraped BBRef playoff stats for 2021-22 through 2024-25
- Calculated BPM delta (playoff vs regular season)
- Built 3-component pressure score (delta + absolute + experience)
- Updated both `CVI_SCORE` and `CVI_PLAYER_SCORE` with playoff context
- Preserved original scores for comparison

Final model uses `CVI_SCORE_FINAL` as the primary score.